In [0]:
%sql
--Analisis de distribucion por tipo de operacion 
SELECT
  tipo_de_operacion,
  COUNT(*) AS frecuencia,
  ROUND(COUNT(*) * 100.00 / SUM(COUNT(*)) OVER (),2)AS porcentaje
FROM bootcamp_matias.raw.propiedades_bronze
GROUP BY 
  tipo_de_operacion
ORDER BY 
  porcentaje DESC

-- claramente el mercado inmobiliario es dominado por ventas con el 62.98% de las propiedades, mientras que el 36.35% son alquileres

/*
Problemas de calidad:

alquier -> error tipografico -> alquiler
alquiler_temporario, alquiler_temporal -> variante de escritura -> alquiler_temporal
venta/alquiler, venta y alquiler,venta_y_alquiler -> dato incosistente -> venta_alquiler 
null -> filtrar
*/

In [0]:
%sql

--Analisis de dsitribucion por moneda
SELECT
  moneda,
  COUNT(*) AS frecuencia,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),2) AS porcentaje
FROM bootcamp_matias.raw.propiedades_bronze
GROUP BY 
  moneda
ORDER BY 
  porcentaje DESC

/*
El análisis de la variable moneda muestra que la mayoría de las transacciones se registran en dólares  (71,9%), seguido por el peso argentino (27,8%). Otras monedas como el peso mexicano y el guaraní presentan frecuencias muy bajas. Asimismo, se detectó un pequeño porcentaje de registros sin información de moneda (0,32%), lo que sugiere posibles problemas de calidad de datos.
*/

In [0]:
%sql

--Analisis de distribucion por ambientes
SELECT
  ambientes,
  COUNT(*) AS frecuencia,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),2)AS porcentaje
FROM bootcamp_matias.raw.propiedades_bronze
GROUP BY 
  ambientes
ORDER BY 
  porcentaje DESC 

/*
Más del 70% de las propiedades tienen entre 2 y 4 ambientes.
Las propiedades muy grandes (>6 ambientes) representan una proporción muy baja del dataset.
Esto es consistente con el mercado inmobiliario urbano, donde predominan departamentos pequeños y medianos.
*/

/*
Problema de calidad:
Valores nulos
Valores incosistentes (0, 1.5 / 2.5, 17, 20, 30, 49, 88, 222)
Outliers (10, 20, 222) ambientes
*/

In [0]:
%sql

-- Analisis de distribucion por estado
SELECT
  estado,
  COUNT(*) AS frecuencia,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),2) AS porcentaje
FROM bootcamp_matias.raw.propiedades_bronze
GROUP BY 
  estado
ORDER BY 
  porcentaje DESC

/*
Entre los valores válidos, predominan:
excelente
bueno
  */

  /*
  Problema de calidad:
  valores nulos,
  Valores inconsistentes (a estrenar / a_estrenar, muy buen estado /muy buena )
  Valores invalidos (0, -1, -2)
  
Categoria normalizada	  Valores originales
excelente	,              impecable, excelente
bueno,                   muy buen estado, bueno
regular	                 regular
a_refaccionar	           a_refaccionar
nuevo	a_estrenar,        nuevo
*/

In [0]:
%sql
--Analisis de distribucion por zona
SELECT
  zona,
  COUNT(*) AS frecuencia,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),2) AS porcentaje
FROM bootcamp_matias.raw.propiedades_bronze
GROUP BY 
  zona
ORDER BY 
  porcentaje DESC
--LIMIT 15

/*
Capital Federal concentra casi el 23% del dataset.
Luego aparecen municipios del Gran Buenos Aires.
Esto indica que el dataset está fuertemente concentrado en AMBA.
*/
/*
Problemas de calidad:
Incosistencia en formato (capital-federal, gba-zona-norte--tigre, bsas-gba-norte/tigre)
Redundacia (moron, gba-zona-oeste--moron, bsas-gba-oeste/moron)
*/

In [0]:
--dim_zona
SELECT
  zona,
  REGEXP_EXTRACT(zona,'([a-z]+(?:-[a-z]+)*)$',1) AS ciudad,

  CASE
    WHEN zona = 'capital-federal' THEN 'caba'

    
    WHEN zona RLIKE 'bsas-gba-norte|gba-zona-norte' THEN 'gba_norte'
    WHEN zona RLIKE 'bsas-gba-sur|gba-zona-sur' THEN 'gba_sur'
    WHEN zona RLIKE 'bsas-gba-oeste|gba-zona-oeste' THEN 'gba_oeste'


    WHEN REGEXP_EXTRACT(zona,'([a-z]+(?:-[a-z]+)*)$',1)
         RLIKE 'tigre|pilar|vicente-lopez|san-isidro|escobar|san-fernando|san-miguel|jose-c-paz|malvinas-argentinas|general-san-martin'
         THEN 'gba_norte'

    WHEN REGEXP_EXTRACT(zona,'([a-z]+(?:-[a-z]+)*)$',1)
         RLIKE 'quilmes|lanus|avellaneda|lomas-de-zamora|berazategui|florencio-varela|almirante-brown|esteban-echeverria|ezeiza|presidente-peron|san-vicente|la-plata|ensenada|berisso'
         THEN 'gba_sur'

    WHEN REGEXP_EXTRACT(zona,'([a-z]+(?:-[a-z]+)*)$',1)
         RLIKE 'moron|la-matanza|ituzaingo|hurlingham|merlo|moreno|tres-de-febrero|general-rodriguez|lujan|marcos-paz|canuelas|castelar|caseros'
         THEN 'gba_oeste'

    ELSE 'otros'
  END AS zona

FROM bootcamp_matias.raw.propiedades_bronze
GROUP BY zona